In [3]:
import os
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

# --- CONFIGURATION ---
ROOT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- MAPPING BRAIN ---
LABEL_MAP = {
    # --- BANGLADESHI DATASET ---
    'rohu': 'rohu', 'black rohu': 'rohu', 'catla': 'catla', 'mrigal': 'mrigal',
    'nile tilapia': 'tilapia', 'silver carp': 'silver_carp', 'common carp': 'common_carp',
    'grass carp': 'grass_carp', 'mirror carp': 'mirror_carp',
    'freshwater shark': 'catfish', 'long-whiskered catfish': 'catfish', 'striped catfish': 'catfish',
    
    # --- FRESHNESS DATASET (FFE/Classification) ---
    'fresh_eyes': 'fresh_eye',
    'fresh_gills': 'fresh_gill',
    'nonfresh_eyes': 'stale_eye',
    'nonfresh_gills': 'stale_gill',
    
    # --- SHRIMP DISEASE ---
    'healthy': 'healthy',
    'wssv': 'white_spot_virus', 'wssv_bg': 'white_spot_virus',
    'bg': 'black_gill_disease'
}

# --- UTILS ---
def get_freshness_score(label):
    if 'fresh' in label and 'non' not in label and 'stale' not in label: return 1.0
    if 'stale' in label or 'nonfresh' in label: return 0.0
    if 'healthy' in label: return 1.0
    if 'virus' in label or 'disease' in label: return 0.0
    return -1.0

# --- THE ROBUST HUNTER ENGINE ---
metadata = []

SCAN_TARGETS = [
    ("bangaladeshi-fish-dataset", "species_market"),
    ("fish-classification-dataset", "freshness_quality"),
    ("shrimp-disease-image-bd", "shrimp_disease"),
    ("fish-recognition-ground-truth-data", "wild_background"),
    ("fishnet-dataset", "wild_background_benchmark")
]

print("🚀 Starting Hunter-Seeker (Case-Insensitive Mode)...")

for dataset_name, source_tag in SCAN_TARGETS:
    print(f"\n🔎 Hunting in: {dataset_name}...")
    
    # 1. Find Dataset Root
    found_roots = list(ROOT_DIR.glob(f"**/{dataset_name}"))
    if not found_roots:
        # Try exact match if glob fails
        search_root = ROOT_DIR / dataset_name
    else:
        search_root = found_roots[0]
        
    if not search_root.exists():
        print(f"   ⚠️ Path not found: {dataset_name}")
        continue

    # 2. ROBUST FILE SCANNER (Case Insensitive)
    # This finds .jpg, .JPG, .png, .PNG, .jpeg, .JPEG
    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
    all_files = [
        p for p in search_root.rglob("*") 
        if p.suffix.lower() in valid_extensions
    ]
                 
    count = 0
    debug_folders_seen = set()
    
    for img_path in tqdm(all_files, desc=f"Parsing {dataset_name}", leave=False):
        folder_name = img_path.parent.name.lower()
        debug_folders_seen.add(folder_name)
        
        final_label = None
        final_type = 'unknown'
        
        # A. Direct Map Match
        for key, val in LABEL_MAP.items():
            if key in folder_name:
                final_label = val
                if source_tag == 'species_market': final_type = 'species'
                elif source_tag == 'freshness_quality': final_type = 'freshness'
                elif source_tag == 'shrimp_disease': 
                    final_type = 'disease'
                    if val == 'healthy': final_type = 'disease'
                break
        
        # B. Wild Background Fallback
        if final_label is None and "wild_background" in source_tag:
            if count > 5000: continue 
            final_label = 'wild_fish_background'
            final_type = 'species'
            
        if final_label:
            metadata.append({
                'path': str(img_path),
                'type': final_type,
                'label': final_label,
                'freshness_score': get_freshness_score(final_label),
                'source': source_tag,
                'original_folder': folder_name
            })
            count += 1
            
    print(f"   ✅ Captured {count} images.")
    if count == 0:
        print(f"   ❌ DEBUG: Saw these folders but matched none: {list(debug_folders_seen)[:10]}")

# --- EXPORT ---
if len(metadata) > 0:
    df = pd.DataFrame(metadata)
    save_path = OUTPUT_DIR / "metadata_kaggle_fused.csv"
    df.to_csv(save_path, index=False)
    
    print("\n🏆 FUSION COMPLETE")
    print(f"Total Images Indexed: {len(df)}")
    print("\n--- Class Breakdown (Top 20) ---")
    print(df['label'].value_counts().head(20))
else:
    print("❌ CRITICAL FAILURE: No images indexed.")

🚀 Starting Hunter-Seeker (Case-Insensitive Mode)...

🔎 Hunting in: bangaladeshi-fish-dataset...


Parsing bangaladeshi-fish-dataset:   0%|          | 0/4389 [00:00<?, ?it/s]

   ✅ Captured 4389 images.

🔎 Hunting in: fish-classification-dataset...


Parsing fish-classification-dataset:   0%|          | 0/3926 [00:00<?, ?it/s]

   ✅ Captured 3926 images.

🔎 Hunting in: shrimp-disease-image-bd...


...

   ✅ Captured 3149 images.

🔎 Hunting in: fish-recognition-ground-truth-data...


Parsing fish-recognition-ground-truth-data:   0%|          | 0/54740 [00:00<?, ?it/s]

   ✅ Captured 5001 images.

🔎 Hunting in: fishnet-dataset...


Parsing fishnet-dataset:   0%|          | 0/94805 [00:00<?, ?it/s]

   ✅ Captured 5001 images.

🏆 FUSION COMPLETE
Total Images Indexed: 21466

--- Class Breakdown (Top 20) ---
label
wild_fish_background    10002
fresh_gill               2274
fresh_eye                1652
white_spot_virus         1548
healthy                   903
rohu                      820
black_gill_disease        698
catfish                   581
common_carp               517
catla                     432
mirror_carp               415
grass_carp                410
tilapia                   409
mrigal                    405
silver_carp               400
Name: count, dtype: int64


In [4]:
import os
import cv2
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from sklearn.model_selection import StratifiedGroupKFold

# --- CONFIGURATION ---
INPUT_CSV = Path("/kaggle/working/processed/metadata_kaggle_fused.csv")
OUTPUT_IMG_DIR = Path("/kaggle/working/processed/images_224")
OUTPUT_IMG_DIR.mkdir(parents=True, exist_ok=True)
IMG_SIZE = 224

# --- ENGINE ---
def compute_md5(img_array):
    return hashlib.md5(img_array.tobytes()).hexdigest()

if not INPUT_CSV.exists():
    print("⚠️ HALT: Run Notebook 01 first!")
else:
    df = pd.read_csv(INPUT_CSV)
    print(f"📥 Loaded War Map: {len(df)} candidate images.")

    valid_records = []
    
    print(f"⚡ processing images (Resize {IMG_SIZE}x{IMG_SIZE} + MD5 Hash)...")
    
    # Iterate with Progress Bar
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        try:
            # 1. Read Image
            img_path = row['path']
            img = cv2.imread(img_path)
            
            if img is None:
                continue # Skip corrupt/unreadable files

            # 2. Resize (Standardize for MobileNetV2)
            img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            
            # 3. Compute Fingerprint (MD5)
            content_hash = compute_md5(img_resized)
            
            # 4. Save Clean Copy (Fast I/O name)
            # We use hash as filename to auto-dedupe on write
            new_filename = f"{content_hash}.jpg"
            new_path = OUTPUT_IMG_DIR / new_filename
            
            if not new_path.exists():
                cv2.imwrite(str(new_path), img_resized)
            
            # 5. Update Record
            record = row.to_dict()
            record['new_path'] = str(new_path)
            record['content_hash'] = content_hash
            record['stratify_group'] = f"{row['type']}_{row['label']}" # For balanced splitting
            valid_records.append(record)
            
        except Exception as e:
            # Silent fail on bad images to keep pipeline moving
            pass

    # --- DEDUPLICATION LOGIC ---
    clean_df = pd.DataFrame(valid_records)
    initial_count = len(clean_df)
    
    # Drop Exact Duplicates based on Hash (Keep first instance)
    # This kills "Augmented vs Original" exact copies
    clean_df = clean_df.drop_duplicates(subset=['content_hash'])
    final_count = len(clean_df)
    
    print(f"\n💀 Nuked {initial_count - final_count} duplicates.")
    print(f"✅ Final Unique Dataset: {final_count} images.")

    # --- THE LEAK-PROOF SPLIT ---
    # We use StratifiedGroupKFold.
    # Groups = content_hash (Ensures copies don't leak if they somehow survived)
    # Stratify = Label (Ensures balanced classes)
    
    print("✂️ Performing Stratified Group Split...")
    sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
    
    # We only need one fold for Val (10% split)
    clean_df['fold'] = -1
    for fold, (train_idx, val_idx) in enumerate(sgkf.split(clean_df, clean_df['stratify_group'], clean_df['content_hash'])):
        clean_df.loc[val_idx, 'fold'] = fold
    
    # Fold 0 is Validation, Folds 1-9 are Training
    val_df = clean_df[clean_df['fold'] == 0].copy()
    train_df = clean_df[clean_df['fold'] != 0].copy()
    
    # Export
    train_df.to_csv("/kaggle/working/processed/train_final.csv", index=False)
    val_df.to_csv("/kaggle/working/processed/val_final.csv", index=False)
    
    print("\n🎯 SPLIT REPORT:")
    print(f"Train Set: {len(train_df)} images")
    print(f"Val Set:   {len(val_df)} images")
    print("-" * 30)
    print("Validation Class Balance:")
    print(val_df['label'].value_counts().head(10))
    print("-" * 30)
    print("Ready for Notebook 03 (Training).")

📥 Loaded War Map: 21466 candidate images.
⚡ processing images (Resize 224x224 + MD5 Hash)...


  0%|          | 0/21466 [00:00<?, ?it/s]


💀 Nuked 206 duplicates.
✅ Final Unique Dataset: 21260 images.
✂️ Performing Stratified Group Split...


KeyError: '[105, 176, 719, 884, 1250, 1366, 1376, 2057, 2074, 2093, 2103, 2112, 2225, 2226, 2505, 2550, 3906, 8594, 11630, 17161, 18077] not in index'